# Exploring `semianalysisai/cc-traces-weka-061326`

WekaTrace corpus derived from SemiAnalysis Claude Code proxy traces (v7, Jun 13 2026).

**Structure** — `traces.jsonl` holds one JSON object per *session* (183 rows):

| field | meaning |
|---|---|
| `id` | session uuid |
| `models` | model names seen in the session |
| `block_size` | KV prefix-cache block size (64) |
| `hash_id_scope` | scope of `hash_ids` (`local` = per-session) |
| `requests` | list of model requests in the session |

Each **request**: `t` (relative start time, s), `model`, `in`/`out` (input/output tokens), `hash_ids` (ordered list of 64-token KV prefix-block hashes), `api_time` (s), `type` (single char), `ttft` (s).

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

REPO = "semianalysisai/cc-traces-weka-061326"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

Matplotlib is building the font cache; this may take a moment.
/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the traces (cached locally by the HF hub)

In [2]:
path = hf_hub_download(REPO, "traces.jsonl", repo_type="dataset")
print("cached at:", path)

sessions = []
with open(path) as f:
    for line in f:
        sessions.append(json.loads(line))
print(f"{len(sessions)} sessions")
print(f"{sum(len(s['requests']) for s in sessions):,} total requests")

cached at: /Users/robertshaw/.cache/huggingface/hub/datasets--semianalysisai--cc-traces-weka-061326/snapshots/c66d8fdaf812342a1464820f22f53ccdfcfaff3a/traces.jsonl
183 sessions
27,300 total requests


## 2. Flatten to a request-level DataFrame

In [10]:
sessions[0].keys()

dict_keys(['id', 'models', 'block_size', 'hash_id_scope', 'requests'])

In [43]:
print(sessions[2]["requests"][1]["hash_ids"])

[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/r

In [34]:
len(sessions[10]["requests"][1]["hash_ids"])*64

44800

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/robertshaw/llm-d/aiperf-bench/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/r

In [28]:
print(sessions[10]["requests"][1]["hash_ids"])

[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 

In [13]:
rows = []
for s in sessions:
    for i, r in enumerate(s["requests"]):
        rows.append({
            "session": s["id"],
            "req_idx": i,
            "t": r["t"],
            "models": s["models"],
            "in": r["in"],
            "out": r["out"],
            "api_time": r.get("api_time"),
            "ttft": r.get("ttft"),
            "type": r.get("type"),
            "n_blocks": len(r.get("hash_ids", [])),
        })
df = pd.DataFrame(rows)
print(df.shape)
df.head()

KeyError: 'in'

## 3. High-level distributions

In [ ]:
df[["in", "out", "api_time", "ttft", "n_blocks"]].describe(percentiles=[.5, .9, .99]).round(2)

In [ ]:
print("By model:")
display(df.groupby("model").agg(
    n=("in", "size"),
    in_med=("in", "median"),
    out_med=("out", "median"),
    ttft_med=("ttft", "median"),
).round(2))

print("\nBy request type:")
display(df.groupby("type").agg(
    n=("in", "size"),
    in_med=("in", "median"),
    out_med=("out", "median"),
).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log10(df["in"].clip(lower=1)), bins=60, color="steelblue")
axes[0].set_title("Input tokens (log10)"); axes[0].set_xlabel("log10(in)")
axes[1].hist(np.log10(df["out"].clip(lower=1)), bins=60, color="indianred")
axes[1].set_title("Output tokens (log10)"); axes[1].set_xlabel("log10(out)")
plt.tight_layout(); plt.show()

## 4. Session-level view (multi-turn shape & KV-cache reuse potential)

In [ ]:
sess = df.groupby("session").agg(
    n_requests=("in", "size"),
    duration_s=("t", "max"),
    total_in=("in", "sum"),
    total_out=("out", "sum"),
    n_models=("model", "nunique"),
).round(1)
sess.describe(percentiles=[.5, .9, .99]).round(1)

In [ ]:
# Prefix-cache reuse within a session: how often do hash blocks repeat across requests?
def reuse_stats(session):
    seen = set()
    total = reused = 0
    for r in session["requests"]:
        for h in r.get("hash_ids", []):
            total += 1
            if h in seen:
                reused += 1
            else:
                seen.add(h)
    return pd.Series({"blocks": total, "reuse_frac": reused / total if total else 0.0})

reuse = pd.DataFrame([reuse_stats(s) for s in sessions])
print(f"Median within-session prefix-block reuse: {reuse['reuse_frac'].median():.1%}")
reuse["reuse_frac"].plot.hist(bins=30, figsize=(8, 3), title="Within-session KV prefix-block reuse fraction")
plt.xlabel("reuse fraction"); plt.show()

## 5. Scratchpad

`sessions` (list of dicts) and `df` (request-level frame) are in scope — explore from here.